# 0. Environnement

In [ ]:
import io
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

BUCKET = "projet-mesure-qualite-rp"
CAMPAGNE = "RG25"
RACINE = f"{BUCKET}/data/raw/{CAMPAGNE}/BIE"

SORTIE = Path("vignettes")          # local, ignoré par git
SORTIE.mkdir(exist_ok=True)
     

import s3fs

fs = s3fs.S3FileSystem()
fichiers = fs.find(RACINE)
print(len(fichiers), "images dans la BIE")
     



# 1. Inventaire

Le nom d'une image suit NUMUT_CABARRES_SUFFIXE.jpg. Le suffixe indique la zone. Le chemin porte le lot, le département, la commune et la zone de collecte.


In [ ]:


df = pd.DataFrame({"chemin": fichiers})
df["nom"] = df.chemin.map(lambda c: Path(c).stem)

morceaux = df.nom.str.split("_")
df["numut"] = morceaux.str[0]
df["cabarres"] = morceaux.str[1]
df["suffixe"] = morceaux.str[2]

parties = df.chemin.str.split("/")
df["lot"] = parties.str[5]
df["dep"] = parties.str[6]
df["com"] = parties.str[7]
df["zc"] = parties.str[8]

# identifiant complet : le code a barres seul n'est pas unique
# entre departements, il faut son contexte geographique
df["doc_id"] = (df.lot + "/" + df.dep + "/" + df.com + "/"
                + df.zc + "/" + df.numut + "_" + df.cabarres)

df.head(3)
     

print(df.suffixe.value_counts().to_string())
print()
print(df.doc_id.nunique(), "documents distincts")
print()
print("images par document :")
print(df.groupby("doc_id").size().value_counts().to_string())
     


In [ ]:
df.dep.value_counts().to_frame("images")


# 2. Charger une image de zone LOG

LOG est la page 4 de la feuille de logement, celle qui porte les questions sur les caractéristiques du logement.


In [ ]:
logs = df[df.suffixe == "LOG"].reset_index(drop=True)
print(len(logs), "images LOG")

def charger(chemin):
    with fs.open(chemin, "rb") as f:
        return Image.open(io.BytesIO(f.read())).convert("RGB")

img = charger(logs.chemin[0])
print("taille :", img.size, "| mode :", img.mode)
print("document :", logs.doc_id[0])


# 3. Repérer les coordonnées des six cases

Le formulaire est imprimé : les cases sont toujours au même endroit. On les repère une seule fois, puis on les enregistre en configuration.

La grille ci-dessous donne les coordonnées à la centaine près. On zoome ensuite sur la zone qui nous intéresse pour affiner.


In [ ]:


plt.figure(figsize=(13, 17))
plt.imshow(img)
plt.grid(True, color="red", alpha=0.35, linewidth=0.6)
plt.xticks(range(0, img.size[0], 100), rotation=90, fontsize=7)
plt.yticks(range(0, img.size[1], 100), fontsize=7)
plt.title("Zone LOG — repérage des coordonnées")
plt.tight_layout()
plt.show()
     

# Zoom sur une region, pour affiner. Ajuster les bornes.
X0, Y0, X1, Y1 = 0, 0, 600, 600

plt.figure(figsize=(11, 11))
plt.imshow(img.crop((X0, Y0, X1, Y1)))
plt.grid(True, color="red", alpha=0.4, linewidth=0.6)
plt.xticks(range(0, X1 - X0, 20), labels=range(X0, X1, 20),
           rotation=90, fontsize=6)
plt.yticks(range(0, Y1 - Y0, 20), labels=range(Y0, Y1, 20), fontsize=6)
plt.tight_layout()
plt.show()
     



# Détection automatique des cadres 

Plutôt que de relever 200 variables à la main, on peut détecter les rectangles imprimés par analyse de contours. Le tri final reproduit l'ordre de lecture du questionnaire — de gauche à droite et de haut en bas — qui est précisément l'ordre qu'exige la variable DETAIL.


In [ ]:


try:
    import cv2

    def detecter_cadres(pil_img, aire_min=300, aire_max=4000,
                        ratio=(0.6, 1.8)):
        gris = np.array(pil_img.convert("L"))
        binaire = cv2.adaptiveThreshold(
            gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 25, 10)
        contours, _ = cv2.findContours(
            binaire, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        boites = []
        for c in contours:
            x, y, w, h = cv2.boundingRect(c)
            if aire_min < w * h < aire_max and ratio[0] < w / h < ratio[1]:
                boites.append((x, y, w, h))
        return sorted(boites, key=lambda b: (b[1], b[0]))

    cadres = detecter_cadres(img)
    print(len(cadres), "cadres candidats")
    print(cadres[:12])
except ImportError:
    print("opencv non installé — repérage manuel")
    cadres = []
     

# Visualiser les cadres detectes
if cadres:
    fig, ax = plt.subplots(figsize=(13, 17))
    ax.imshow(img)
    for x, y, w, h in cadres:
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False,
                                   edgecolor="red", linewidth=1))
    ax.set_title(f"{len(cadres)} cadres détectés")
    plt.tight_layout()
    plt.show()
     


In [ ]:
pip install opencv-python-headless

In [ ]:
    fig, ax = plt.subplots(figsize=(16,14))
    ax.imshow(img)
    for i, (x, y, w, h) in enumerate(cadres):
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False,
                                   edgecolor="red", linewidth=1))
        ax.text(x,y-2,str(i),fontsize=5,color="blue")
    plt.axis("off")
    plt.show()
     

In [ ]:
mes_cases=[cadres[i] for i in [16,20,25,27,35,39]]
for lib, box in zip(["Maison", "Appartement", "Logement-foyer",
                          "Chambre d'hôtel", "Habitation de fortune",
                          "Pièce indépendante"],mes_cases):
    print(f"{lib:24}{box}")

# 4. Déclarer les positions en configuration

In [ ]:
POSITIONS = {
    "LOG": {
        "TYPL": {
            "type": "cases",
            "question": "FL_Q1",
            "libelle": "Type de logement",
            "modalites": ["Maison", "Appartement", "Logement-foyer",
                          "Chambre d'hôtel", "Habitation de fortune",
                          "Pièce indépendante"],
            # x, y, largeur, hauteur — dans l'ordre de lecture
            "cases": mes_cases
        },
    },
}

In [ ]:
Path("configs").mkdir(exist_ok=True)
with open("configs/positions_FL_2025.json", "w", encoding="utf-8") as f:
    json.dump(POSITIONS, f, indent=2, ensure_ascii=False)
print("positions enregistrées")

# 5. Découper et vérifier visuellement

Le contrôle visuel n'est pas optionnel. Une case mal cadrée produit des prédictions fausses sans jamais lever d'erreur.


In [ ]:
def decouper(image, boites):
    return [image.crop((x, y, x + w, y + h)) for x, y, w, h in boites]

boites = POSITIONS["LOG"]["TYPL"]["cases"]
vignettes = decouper(img, boites)

fig, axes = plt.subplots(1, len(vignettes), figsize=(2 * len(vignettes), 2.4))
for ax, v, lib in zip(axes, vignettes,
                      POSITIONS["LOG"]["TYPL"]["modalites"]):
    ax.imshow(v)
    ax.set_title(lib, fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()

# 6. Une première perception sans aucun modèle

Avant d'entraîner quoi que ce soit, on mesure ce que donne une règle de densité : la part de pixels sombres dans la case. C'est une référence basse  tout modèle devra faire mieux, et beaucoup d'erreurs se voient dès ce stade.

Cette heuristique sépare raisonnablement vide et cochée. Elle ne distingue pas cochée de biffée —c'est précisément la distinction difficile, celle qui change la valeur de la question, et celle qui justifie un modèle.


In [ ]:
def densite(vignette, marge=0.18, seuil_gris=140):
    """Part de pixels sombres, hors bordure imprimée de la case."""
    a = np.array(vignette.convert("L"), dtype=float)
    h, w = a.shape
    mh, mw = int(h * marge), int(w * marge)
    interieur = a[mh:h - mh, mw:w - mw]
    return float((interieur < seuil_gris).mean())

SEUIL_DENSITE = 0.08      # a calibrer sur des cas connus

def percevoir_densite(vignettes, seuil=SEUIL_DENSITE):
    """Renvoie (etat, score). Reference basse, sans apprentissage."""
    out = []
    for v in vignettes:
        d = densite(v)
        etat = "cochee" if d >= seuil else "vide"
        marge = abs(d - seuil) / max(seuil, 1e-6)
        out.append((etat, float(min(0.99, 0.5 + 0.5 * min(marge, 1.0)))))
    return out

percepts = percevoir_densite(vignettes)
for lib, (etat, score) in zip(POSITIONS["LOG"]["TYPL"]["modalites"], percepts):
    print(f"{lib:24} {etat:8} score {score:.2f}")

In [ ]:
for lib, v in zip(POSITIONS["LOG"]["TYPL"]["modalites"],vignettes):
    print(f"{lib:24}{densite(v):.4f}")

# 7. Appliquer la règle des cases

In [ ]:


FORCE = ("cochee", "biffee", "coloree")
MODALITE_BASSE = {"BI_Q10", "BI_Q11"}
BLANC_SI_PLUSIEURS = {"BI_Q9", "FL_Q3", "FL_Q7", "FL_Q13",
                      "FLDOM_Q3", "FLDOM_Q7", "FLDOM_Q11", "FLDOM_Q14",
                      "TPSRESALT", "AUTRECOMETU"}


def valeur_case(etat, gagnant):
    if etat == "vide":
        return "0"
    if gagnant is not None and etat == gagnant:
        return "1"
    if gagnant == "cochee" and etat == "biffee":
        return "2"
    return "3"


def regle_cases(question, etats):
    """Renvoie (CHOIX, DETAIL)."""
    gagnant = next((f for f in FORCE if f in etats), None)
    detail = "".join(valeur_case(e, gagnant) for e in etats)

    positions = [i + 1 for i, c in enumerate(detail) if c == "1"]
    if not positions:
        return "", detail
    if question in BLANC_SI_PLUSIEURS and len(positions) > 1:
        return "", detail
    if question in MODALITE_BASSE:
        return str(min(positions)), detail
    return str(max(positions)), detail
     

# Les exemples du document de consignes, en tests
assert regle_cases("BI_Q10",
    ["vide", "biffee", "vide", "biffee", "vide", "vide"]) == ("2", "010100")
assert regle_cases("BI_Q13",
    ["cochee", "vide", "cochee", "vide",
     "biffee", "coloree", "vide", "vide"]) == ("3", "10102300")
assert regle_cases("BI_Q9", ["cochee", "cochee"]) == ("", "11")
assert regle_cases("BI_Q31", ["autre", "vide", "vide"]) == ("", "300")
print("les exemples du document passent")
     

etats = [e for e, _ in percepts]
choix, detail = regle_cases(POSITIONS["LOG"]["TYPL"]["question"], etats)
score = min(s for _, s in percepts)   # le champ vaut le minimum de ses vignettes

print(f"états   : {etats}")
print(f"DETAIL  : {detail}")
print(f"CHOIX   : {choix!r}")
print(f"score   : {score:.2f}")
     


# 8. Traiter un lot de documents

In [ ]:
def traiter(chemin, positions, zone, variable):
    image = charger(chemin)
    conf = positions[zone][variable]
    vign = decouper(image, conf["cases"])
    perc = percevoir_densite(vign)
    etats = [e for e, _ in perc]
    choix, detail = regle_cases(conf["question"], etats)
    return {
        "variable": variable,
        "valeur": choix,
        "detail": detail,
        "score": min(s for _, s in perc),
        "etats": "|".join(etats),
    }


N = 200
lignes = []
for _, r in logs.head(N).iterrows():
    try:
        res = traiter(r.chemin, POSITIONS, "LOG", "TYPL")
        res["doc_id"] = r.doc_id
        lignes.append(res)
    except Exception as e:
        print("échec", r.doc_id, type(e).__name__, e)

resultats = pd.DataFrame(lignes)
resultats
     

print(resultats.valeur.value_counts(dropna=False).to_string())
print()
print("score minimum :", resultats.score.min().round(2))
print("champs sous 0.90 :", int((resultats.score < 0.90).sum()), "/", len(resultats))
     


In [ ]:
vides=resultats[resultats.valeur==""]
print(vides[["doc_id","detail","etats","score"]].to_string())

In [ ]:
print(resultats.columns.tolist())

In [ ]:
for k in range(len(vides)):
    doc=vides.iloc[0].doc_id
    chemin=logs[logs.doc_id==doc].chemin.iloc[0]
    im=charger(chemin)
    vs=decouper(im,POSITIONS["LOG"]["TYPL"]["cases"])
    fig,axes=plt.subplots(1,6,figsize=(12,2.2))
    for ax, v in zip(axes,vs):
        ax.imshow(v);
        ax.set_title(f"{densite(v):.3f}",fontsize=8);
        ax.axis("off")
    fig.suptitle(doc,fontsize=9)    
    plt.show()    

# Partie 2 — Parsing du Fiqual et comparaison

 Réutilise les objets déjà en mémoire : resultats, logs, charger, decouper, densite, POSITIONS.

Objectif. Nommer les colonnes du Fiqual, en extraire TYPL, et le comparer aux valeurs produites par la chaîne image → vignettes → perception → règle.

Réserve. Le Fiqual n'est pas la vérité : c'est une seconde mesure, produite par des opérateurs, elle-même faillible. On mesure donc une concordance, pas un taux d'erreur. Seul l'arbitrage du SeRN tranche en cas de désaccord.
## 2.1 Lecture brute du Fiqual

Le fichier mêle plusieurs structures : une feuille de logement n'a ni le même nombre ni le même sens de champs qu'un bulletin individuel. On lit donc chaque ligne comme une chaîne, sans découper.

Le séparateur \x00 n'apparaît jamais dans le fichier : pandas ne découpe rien.


In [ ]:
FIQUAL = f"s3://{BUCKET}/data/raw/{CAMPAGNE}/RP_PMQ_FIQUAL2599.txt"
FICOD = f"s3://{BUCKET}/data/raw/{CAMPAGNE}/FICOD2599V3.txt"


def lire_brut(chemin, encodage="latin-1"):
    """Une ligne du fichier = une chaîne. Aucune interprétation."""
    return pd.read_csv(chemin, sep="\x00", header=None, names=["ligne"],
                       dtype=str, encoding=encodage,
                       keep_default_na=False, engine="python")["ligne"]


brut = lire_brut(FIQUAL)
print(len(brut), "lignes dans le Fiqual")
     


# 2.2 Inventaire

Combien de champs pour chaque type d'enregistrement ? Préalable indispensable : on ne peut pas nommer des colonnes avant de savoir combien il y en a.

Chaque type doit avoir une seule largeur. Deux largeurs pour un même type signalent un fichier irrégulier, à comprendre avant d'aller plus loin.


In [ ]:
def inventaire(serie, sep="|"):
    ch = serie.str.split(sep)
    return pd.crosstab(ch.str[0], ch.str.len())


inv = inventaire(brut)
inv

In [ ]:
# Le Ficod a les memes types mais des largeurs superieures : le prestataire
# de controle ne saisit que les variables du perimetre controle.
inv_ficod = inventaire(lire_brut(FICOD))

comparaison_largeurs = pd.DataFrame({
    "n_fiqual": inv.apply(lambda r: r.idxmax(), axis=1),
    "lignes_fiqual": inv.max(axis=1),
}).join(pd.DataFrame({
    "n_ficod": inv_ficod.apply(lambda r: r.idxmax(), axis=1),
    "lignes_ficod": inv_ficod.max(axis=1),
}), how="outer")
comparaison_largeurs

# 2.3 Isoler la feuille de logement

Type 1 : feuille de logement métropole, celle qui porte TYPL.


In [ ]:
def table_par_type(serie, enr, sep="|"):
    """DataFrame de colonnes numérotées 0..n-1, pour un type donné."""
    ch = serie.str.split(sep)
    sel = ch[ch.str[0] == enr]
    if sel.empty:
        raise ValueError(f"aucune ligne de type {enr!r}")
    return pd.DataFrame(sel.tolist(), dtype=str)


fq = table_par_type(brut, "1")
print("forme :", fq.shape)

# 2.4 Repérer TYPL sans connaître le dessin de fichier

Les colonnes n'ont pas encore de nom. Deux méthodes, à utiliser ensemble : l'une pour trouver, l'autre pour vérifier.

Par le domaine de valeurs. TYPL vaut 1 à 6. On cherche les colonnes dont toutes les valeurs non vides appartiennent à ce domaine. Plusieurs colonnes vont correspondre — beaucoup de variables ont un domaine restreint.

Par l'annexe 1. Elle liste les variables dans l'ordre exact du fichier ; on compte jusqu'à TYPL. C'est elle qui fait foi ; le domaine ne sert qu'à confirmer.


In [ ]:
def colonnes_compatibles(df, domaine, autoriser_vide=True):
    """Colonnes dont toutes les valeurs appartiennent au domaine."""
    admis = {str(v) for v in domaine}
    if autoriser_vide:
        admis.add("")
    return [c for c in df.columns
            if set(df[c].str.strip().unique()) <= admis]


candidats = colonnes_compatibles(fq, range(1, 7))
print("colonnes compatibles avec le domaine 1..6 :", candidats)

In [ ]:
# La distribution aide a reconnaitre la variable.
# TYPL : les modalites 1 (maison) et 2 (appartement) doivent dominer largement,
# les autres (logement-foyer, chambre d'hotel, habitation de fortune,
# piece independante) etre rares.
for c in candidats:
    print(f"--- colonne {c}")
    print(fq[c].str.strip().value_counts(dropna=False).to_string())
    print()
     

In [ ]:
COL_TYPL = 17     # <-- indice de la colonne identifiee ci-dessus

if COL_TYPL is None:
    print("renseigner COL_TYPL avant de continuer")
else:
    print(fq[COL_TYPL].str.strip().value_counts().to_string())

# Le dessin complet, à terme

Identifier une colonne à la main convient pour une démonstration. Pour la suite, le dessin complet doit être inscrit en configuration, à partir de l'annexe 1.

Le contrôle qui évite un décalage silencieux : le nombre de noms déclarés doit être exactement égal au nombre de colonnes lues.


In [ ]:
# A COMPLETER a partir de l'annexe 1, dans l'ordre exact.
COLONNES_FIQUAL_FL = [
    "ENR", "NUMLS", "NUMUT", "IDENT", "CABFL",
    # ... completer jusqu'au nombre de colonnes
]

print(f"{len(COLONNES_FIQUAL_FL)} noms déclarés / {fq.shape[1]} colonnes lues")
if len(COLONNES_FIQUAL_FL) != fq.shape[1]:
    print("→ incomplet : ne pas nommer tant que le compte ne tombe pas")
     


# 2.5 Construire la clé de jointure

Le code à barres identifie le questionnaire papier et nomme les images. Mais l'échantillon couvre plusieurs départements, et rien ne garantit son unicité d'un département à l'autre — c'est ce qui expliquait les documents au nombre d'images irrégulier repérés dans la partie 1.

La clé doit donc être construite de la même façon des deux côtés.


In [ ]:
# Les 8 premieres colonnes, 3 lignes : pour reconnaitre la structure.
fq.iloc[:3, :8]

In [ ]:
POS_NUMUT, POS_CABFL = 2, 4      # <-- a confirmer avec l'annexe 1

fq = fq.assign(cle=fq[POS_NUMUT].str.strip() + "_" + fq[POS_CABFL].str.strip())

print(fq.cle.nunique(), "clés distinctes pour", len(fq), "lignes")
print("doublons :", int(fq.cle.duplicated().sum()))

# 2.6 Comparer

resultats vient de la partie 1 : la valeur de TYPL produite par la chaîne complète. On construit sa clé de la même façon, puis on joint.


In [ ]:
produits = resultats.copy()
produits["cle"] = produits.doc_id.str.split("/").str[-1]
produits["valeur"] = produits.valeur.astype(str).str.strip()

print(len(produits), "valeurs produites")
print(produits.cle.head(3).tolist())
print(fq.cle.head(3).tolist())

In [ ]:
if COL_TYPL is not None:
    ref = fq[["cle", COL_TYPL]].rename(columns={COL_TYPL: "ref"})
    ref["ref"] = ref["ref"].astype(str).str.strip()

    comp = produits.merge(ref, on="cle", how="inner")
    comp["concorde"] = comp.valeur == comp.ref

    print(f"{len(comp)} documents comparés sur {len(produits)} produits")
    if len(comp):
        print(f"concordance : {comp.concorde.mean():.1%}")
else:
    comp = None
    print("renseigner COL_TYPL")
     

# 2.7 Regarder les désaccords

Un taux global ne dit pas quoi corriger. Les cas de désaccord, si.


In [ ]:
if comp is not None and len(comp):
    desaccords = comp[~comp.concorde]
    print(len(desaccords), "désaccords")
    if len(desaccords):
        print()
        print(pd.crosstab(desaccords.valeur, desaccords.ref,
                          rownames=["produit"], colnames=["fiqual"]))

In [ ]:
# Inspection visuelle d'un desaccord.
# charger, decouper, densite et POSITIONS viennent de la partie 1.
if comp is not None and len(comp) and len(desaccords):
    d = desaccords.iloc[0]
    chemin = logs[logs.doc_id == d.doc_id].chemin.iloc[0]
    vs = decouper(charger(chemin), POSITIONS["LOG"]["TYPL"]["cases"])

    fig, axes = plt.subplots(1, len(vs), figsize=(2 * len(vs), 2.4))
    for ax, v, lib in zip(axes, vs,
                          POSITIONS["LOG"]["TYPL"]["modalites"]):
        ax.imshow(v)
        ax.set_title(f"{lib}\n{densite(v):.3f}", fontsize=7)
        ax.axis("off")
    fig.suptitle(f"produit {d.valeur!r} — fiqual {d.ref!r}", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
d=desaccords.iloc[0]
chemin=logs.loc[logs.doc_id==d.doc_id,"chemin"].iloc[0]
print(chemin)

In [ ]:
img_d=charger(chemin)
plt.figure(figsize=(14,18))
plt.imshow(img_d)
plt.axis("off")
plt.show()

In [ ]:
print(len(comp),"documents comparés sur",len(produits),"produits")

# 2.9 Où sont passés les documents non appariés

In [ ]:


apparies = set(comp.cle)
produits["apparie"] = produits.cle.isin(apparies)

print(f"{produits.apparie.sum()} appariés / {len(produits)} produits")
print(f"{(~produits.apparie).sum()} non appariés")
     


Par le département

Le département figure dans le chemin de l'image. Les codes 971 à 978 sont les départements et collectivités d'outre-mer.


In [ ]:


produits["dep"] = produits.doc_id.str.split("/").str[1]

repartition = pd.crosstab(produits.dep, produits.apparie)
repartition.columns = ["non apparié", "apparié"]
repartition
     

DOM = {"971", "972", "973", "974", "975", "976", "977", "978"}

produits["outre_mer"] = produits.dep.isin(DOM)

tab = pd.crosstab(produits.outre_mer, produits.apparie)
tab.index = ["métropole", "outre-mer"]
tab.columns = ["non apparié", "apparié"]
print(tab.to_string())
     


In [ ]:

orphelins_metropole = produits[(~produits.apparie) & (~produits.outre_mer)]

if len(orphelins_metropole):
    print(len(orphelins_metropole), "non appariés en métropole — à comprendre :")
    print(orphelins_metropole[["doc_id", "valeur"]].to_string())
else:
    print("aucun non apparié en métropole : tous les orphelins sont ultramarins")
     


Par le Fiqual

Contrôle réciproque : les documents non appariés se retrouvent-ils dans le type 2 du Fiqual, celui des feuilles de logement d'outre-mer ?

C'est la vérification qui démontre l'hypothèse, au lieu de la déduire du seul département.


In [ ]:


fq2 = table_par_type(brut, "2")
fq2 = fq2.assign(cle=fq2[POS_NUMUT].str.strip() + "_" + fq2[POS_CABFL].str.strip())

print(f"type 2 : {len(fq2)} lignes, {fq2.cle.nunique()} clés distinctes")
     


In [ ]:
non_apparies = produits.loc[~produits.apparie, "cle"]
dans_type2 = non_apparies.isin(set(fq2.cle))

print(f"{dans_type2.sum()} / {len(non_apparies)} retrouvés dans le type 2")

restants = non_apparies[~dans_type2]
if len(restants):
    print(f"\n{len(restants)} introuvables dans les types 1 et 2 :")
    print(restants.head(10).to_string())
else:
    print("→ tous les non-appariés sont des feuilles d'outre-mer")

In [ ]:
import io
from pathlib import Path

import pandas as pd
import s3fs

BUCKET = "projet-mesure-qualite-rp"
CAMPAGNES = ["RG25", "RG26"]

fs = s3fs.S3FileSystem()

In [ ]:
for c in CAMPAGNES:
    print(f"--- {c}")
    try:
        for chemin in sorted(fs.ls(f"{BUCKET}/data/raw/{c}")):
            print("   ", chemin.split("/")[-1])
    except FileNotFoundError:
        print("    dossier absent")
    print()
     

In [ ]:
def inventorier(campagne):
    """Renvoie un DataFrame décrivant chaque image de la BIE."""
    fichiers = fs.find(f"{BUCKET}/data/raw/{campagne}/BIE")
    if not fichiers:
        return None

    d = pd.DataFrame({"chemin": fichiers})
    d["nom"] = d.chemin.map(lambda c: Path(c).stem)

    morceaux = d.nom.str.split("_")
    d["numut"] = morceaux.str[0]
    d["cabarres"] = morceaux.str[1]
    d["suffixe"] = morceaux.str[2]

    parties = d.chemin.str.split("/")
    d["lot"] = parties.str[5]
    d["dep"] = parties.str[6]
    d["com"] = parties.str[7]
    d["zc"] = parties.str[8]

    d["doc_id"] = (d.lot + "/" + d.dep + "/" + d.com + "/"
                   + d.zc + "/" + d.numut + "_" + d.cabarres)
    d["campagne"] = campagne
    return d


bases = {}
for c in CAMPAGNES:
    b = inventorier(c)
    if b is None:
        print(f"{c} : pas de BIE")
    else:
        bases[c] = b
        print(f"{c} : {len(b)} images")
     

In [ ]:
lignes = []
for c, b in bases.items():
    lignes.append({
        "campagne": c,
        "images": len(b),
        "documents": b.doc_id.nunique(),
        "départements": b.dep.nunique(),
        "communes": b.com.nunique(),
    })

synthese = pd.DataFrame(lignes).set_index("campagne")
synthese

In [ ]:


zones = pd.DataFrame({c: b.suffixe.value_counts() for c, b in bases.items()})
zones = zones.fillna(0).astype(int)
zones
     


In [ ]:
for c, b in bases.items():
    t = b.groupby("doc_id").size().value_counts().sort_index()
    print(f"--- {c}")
    for n_images, n_docs in t.items():
        note = ""
        if n_images == 7:
            note = "  ← feuilles de logement complètes"
        elif n_images == 1:
            note = "  ← bulletins et cas particuliers"
        else:
            note = "  ← À VÉRIFIER"
        print(f"  {n_docs:5} documents à {n_images:3} image(s){note}")
    print()

In [ ]:


geo = pd.DataFrame({c: b.dep.value_counts() for c, b in bases.items()})
geo = geo.fillna(0).astype(int)

DOM = {"971", "972", "973", "974", "975", "976", "977", "978"}
geo["zone"] = ["outre-mer" if d in DOM else "métropole" for d in geo.index]
geo
     

for c in bases:
    part = geo.loc[geo.zone == "outre-mer", c].sum() / geo[c].sum()
    print(f"{c} : {part:.0%} des images en outre-mer")
     


In [ ]:


for c in CAMPAGNES:
    chemin = f"{BUCKET}/data/raw/{c}/Divergences"
    print(f"--- {c}")
    try:
        for f in sorted(fs.ls(chemin)):
            info = fs.info(f)
            print(f"    {f.split('/')[-1]}  ({info['size'] / 1024:.0f} Ko)")
    except FileNotFoundError:
        print("    pas de dossier Divergences")
    print()
     


In [ ]:


# Lecture du fichier de divergences, si present.
# Colonnes attendues : LIBELLE, VARIABLE, LS, UT, DEP, COMMUNE, ZC,
# RANG_ADR, RANG_LOG, RANG_INDIV, ECHANTILLON, NOM_IMAGE, RESULTAT,
# VALEUR_PAD, VALEUR_PMQ, VALEUR_ATTENDUE

CHEMIN_DIV = f"s3://{BUCKET}/data/raw/RG26/Divergences/divergencesLS99.xlsx"

try:
    div = pd.read_excel(CHEMIN_DIV, dtype=str)
    print(f"{len(div)} divergences, {div.shape[1]} colonnes")
    print()
    print(div.columns.tolist())
except Exception as e:
    div = None
    print("lecture impossible :", type(e).__name__, e)
     

# Sur quelles variables portent les divergences ?
if div is not None and "VARIABLE" in div.columns:
    print(div.VARIABLE.value_counts().head(20).to_string())
     


In [ ]:
# La valeur arbitree est-elle toujours renseignee ?
if div is not None and "VALEUR_ATTENDUE" in div.columns:
    renseignee = div.VALEUR_ATTENDUE.notna() & (div.VALEUR_ATTENDUE.str.strip() != "")
    print(f"{renseignee.sum()} / {len(div)} divergences arbitrées")